In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install tensorflow numpy matplotlib seaborn scikit-learn pandas

In [ ]:
import os
from PIL import Image

DATASET_DIR = "/content/drive/MyDrive/Bayile/Coffee/coffee-dataset"
bad_files = 0

print("Scanning dataset for corrupted images... (This takes a few minutes)")
for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            filepath = os.path.join(root, file)
            try:
                # Try to open the image to see if it's valid
                with Image.open(filepath) as img:
                    img.verify()
            except (IOError, SyntaxError) as e:
                print(f"❌ Deleting corrupted file: {filepath}")
                os.remove(filepath) # Deletes the bad image
                bad_files += 1

print(f"\n✅ Cleanup complete. Removed {bad_files} corrupted images.")

Scanning dataset for corrupted images... (This takes a few minutes)

✅ Cleanup complete. Removed 0 corrupted images.


In [ ]:
!pip install imagehash

In [ ]:
!pip install imagehash

In [ ]:
# ============================================================
# SCRIPT 1 - FF-CNN only
# Colab first cell:  !pip install imagehash
# Only additions: saves test predictions + test indices so Script 2
# can run McNemar against FF-CNN on the identical test set.
# ============================================================
import os, re, gc, json, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image
import imagehash
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, f1_score, log_loss)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers, Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.utils import to_categorical

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
DATASET_DIR   = "/content/local_coffee_dataset"
OUTPUT_DIR    = "Paper_Results_v2"
SEED          = 42
BATCH_SIZE    = 32
LABEL_SMOOTH  = 0.1

# --- grouping (all copies of one photo = one group) ---
GROUP_MODE    = "hash"   # "hash" | "filename" | "both"
HASH_THRESH   = 3        # 0-3. Lower = less aggressive merging. Check group_check.png!
AUG_SUFFIX_RE = r"([_\-\s]?(aug|augmented|rot|rotate|rotated|flip|flipped|mirror|noise|bright|dark|crop|zoom|shift)[_\-\s]?\d*)+$"

# --- cross-validation ---
N_FOLDS       = 5
FOLDS_TO_RUN  = 1        # set to 5 for mean +/- std in the paper

# --- training ---
EPOCHS             = 40
PATIENCE           = 8        # early stopping on validation macro-F1
LR_PATIENCE        = 3        # reduce LR after this many epochs without improvement
SAMPLES_PER_GROUP  = 16       # images drawn per original photo per epoch (stops memorising copies)

SIZES = {"FF-CNN": 224}

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
rng = np.random.default_rng(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. COLLECT FILES
# ------------------------------------------------------------
print("Gathering image paths and labels...")
class_names = sorted(d for d in os.listdir(DATASET_DIR)
                     if os.path.isdir(os.path.join(DATASET_DIR, d)))
NUM_CLASSES = len(class_names)
filepaths, labels = [], []
for i, cn in enumerate(class_names):
    cdir = os.path.join(DATASET_DIR, cn)
    for f in sorted(os.listdir(cdir)):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            filepaths.append(os.path.join(cdir, f))
            labels.append(i)
filepaths = np.array(filepaths)
labels = np.array(labels)
N = len(filepaths)
print(f"{N} images, {NUM_CLASSES} classes: {class_names}")
print("Class counts:", {c: int(n) for c, n in zip(class_names, np.bincount(labels))})

# ------------------------------------------------------------
# 2. GROUPING
# ------------------------------------------------------------
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra

def orientation_hashes(path):
    """pHash under all 8 flips/rotations (index 0 = original)."""
    img = Image.open(path).convert("L")
    img.draft("L", (128, 128))
    arr = np.array(img.resize((64, 64)))
    out = []
    for flip in (False, True):
        base = np.fliplr(arr) if flip else arr
        for k in range(4):
            h = imagehash.phash(Image.fromarray(np.rot90(base, k).copy()))
            out.append(int(str(h), 16))
    return out

HASH_CACHE = os.path.join(OUTPUT_DIR, "phash_cache.npy")
OLD_CACHE = "Paper_Results/phash_cache.npy"

def compute_all_hashes():
    for cache in (HASH_CACHE, OLD_CACHE):
        if os.path.exists(cache):
            H = np.load(cache, allow_pickle=False)
            if H.shape[0] == N:
                print("Loaded cached hashes from", cache)
                return H
    print("Computing perceptual hashes (one-time, a few minutes)...")
    H = np.zeros((N, 8), dtype=np.uint64)
    for i, p in enumerate(filepaths):
        try:
            H[i] = np.array(orientation_hashes(p), dtype=np.uint64)
        except Exception as e:
            print("Hash failed:", p, e)
            H[i] = np.uint64(i)
        if (i + 1) % 5000 == 0:
            print(f"  {i+1}/{N}")
    np.save(HASH_CACHE, H)
    return H

def popcount(x): return bin(x).count("1")

def hash_union(uf):
    assert HASH_THRESH <= 3, "4x16-bit chunk indexing is only exact for Hamming <= 3"
    H = compute_all_hashes().tolist()
    base = [row[0] for row in H]
    index = [dict() for _ in range(4)]
    for j, b in enumerate(base):
        for c in range(4):
            index[c].setdefault((b >> (16 * c)) & 0xFFFF, []).append(j)
    for i in range(N):
        for h in H[i]:
            for c in range(4):
                cand = index[c].get((h >> (16 * c)) & 0xFFFF, [])
                if len(cand) > 2000:
                    continue
                for j in cand:
                    if j != i and labels[j] == labels[i] and popcount(h ^ base[j]) <= HASH_THRESH:
                        uf.union(i, j)
    # NOTE: we only merge within the same class, so a visually similar pair across
    # classes can never glue two classes' groups together.

def filename_union(uf):
    keys = {}
    for i, p in enumerate(filepaths):
        stem = os.path.splitext(os.path.basename(p))[0].lower()
        stem = re.sub(AUG_SUFFIX_RE, "", stem)
        key = (labels[i], stem)
        if key in keys: uf.union(i, keys[key])
        else: keys[key] = i

uf = UnionFind(N)
if GROUP_MODE in ("hash", "both"):     hash_union(uf)
if GROUP_MODE in ("filename", "both"): filename_union(uf)
roots = np.array([uf.find(i) for i in range(N)])
_, groups = np.unique(roots, return_inverse=True)
N_GROUPS = int(groups.max() + 1)
print(f"{N} images -> {N_GROUPS} groups (avg {N/N_GROUPS:.2f} images per original photo)")

# ------------------------------------------------------------
# 2b. GROUP DIAGNOSTICS  (look at group_check.png!)
# ------------------------------------------------------------
sizes = pd.Series(groups).value_counts()
print("\nGroup size summary:\n", sizes.describe().round(1).to_string())
print("Largest 10 groups:", sizes.head(10).tolist())
g_class = pd.DataFrame({"g": groups, "y": labels}).drop_duplicates("g")["y"].values
print("Groups per class:", {c: int(n) for c, n in zip(class_names, np.bincount(g_class, minlength=NUM_CLASSES))})
sizes.rename("n_images").to_csv(f"{OUTPUT_DIR}/group_sizes.csv")

def save_group_montage(n_big=3, n_mid=3, per_group=6):
    ids = list(sizes.index[:n_big])
    mid = sizes.index[len(sizes)//3: 2*len(sizes)//3]
    ids += list(rng.choice(mid, size=min(n_mid, len(mid)), replace=False))
    fig, axes = plt.subplots(len(ids), per_group, figsize=(per_group*2, len(ids)*2.1))
    axes = np.atleast_2d(axes)
    for r, g in enumerate(ids):
        members = np.where(groups == g)[0]
        pick = rng.choice(members, size=min(per_group, len(members)), replace=False)
        for c in range(per_group):
            ax = axes[r, c]; ax.axis("off")
            if c < len(pick):
                ax.imshow(Image.open(filepaths[pick[c]]).convert("RGB").resize((160, 160)))
                if c == 0:
                    ax.set_title(f"group {g}: {len(members)} imgs, {class_names[labels[members[0]]]}",
                                 fontsize=7, loc="left")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/group_check.png", dpi=130)
    plt.close()
save_group_montage()
print("Saved group_check.png - rows should be the SAME leaf photo, not different leaves.\n")

# ------------------------------------------------------------
# 3. SPLITS (grouped + stratified): train / val / test
# ------------------------------------------------------------
outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = []
for trval_idx, test_idx in outer.split(filepaths, labels, groups):
    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
    tr_rel, va_rel = next(inner.split(filepaths[trval_idx], labels[trval_idx], groups[trval_idx]))
    folds.append((trval_idx[tr_rel], trval_idx[va_rel], test_idx))

def check_no_leak(tr, va, te):
    g_tr, g_va, g_te = set(groups[tr]), set(groups[va]), set(groups[te])
    assert not (g_tr & g_te) and not (g_tr & g_va) and not (g_va & g_te), "Group leakage between splits!"

def split_report(name, idx):
    gc_ = np.bincount(g_class[np.unique(groups[idx])], minlength=NUM_CLASSES)
    ic_ = np.bincount(labels[idx], minlength=NUM_CLASSES)
    print(f"  {name:5s} images/class={ic_.tolist()}  groups/class={gc_.tolist()}")

# ------------------------------------------------------------
# 4. DATA PIPELINE (raw 0-255; every model normalises internally)
# ------------------------------------------------------------
def make_ds(idx, size, train=False, sample_weight=None):
    y = to_categorical(labels[idx], NUM_CLASSES)
    tensors = (filepaths[idx], y) if sample_weight is None else (filepaths[idx], y, sample_weight.astype("float32"))
    ds = tf.data.Dataset.from_tensor_slices(tensors)
    if train:
        ds = ds.shuffle(len(idx), reshuffle_each_iteration=True)

    def load(path, *rest):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, (size, size))
        return (tf.cast(img, tf.float32), *rest)

    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def sample_epoch_indices(train_idx):
    """Draw at most SAMPLES_PER_GROUP images from every training group."""
    g = groups[train_idx]
    order = np.argsort(g, kind="stable")
    g_sorted = g[order]
    bounds = np.flatnonzero(np.r_[True, g_sorted[1:] != g_sorted[:-1], True])
    chosen = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        members = train_idx[order[a:b]]
        k = min(SAMPLES_PER_GROUP, len(members))
        chosen.append(rng.choice(members, size=k, replace=False))
    return np.concatenate(chosen)

# ------------------------------------------------------------
# 5. MODEL (FF-CNN)
# ------------------------------------------------------------
loss_fn = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)

class RandomCutout(layers.Layer):
    """Randomly blanks a square patch (training only). Input range 0-255."""
    def __init__(self, frac=0.25, prob=0.5, **kw):
        super().__init__(**kw); self.frac, self.prob = frac, prob
    def call(self, x, training=None):
        if not training: return x
        b = tf.shape(x)[0]; h = tf.shape(x)[1]; w = tf.shape(x)[2]
        half = tf.cast(tf.cast(h, tf.float32) * self.frac / 2, tf.int32)
        cy = tf.random.uniform([b], 0, h, tf.int32)
        cx = tf.random.uniform([b], 0, w, tf.int32)
        ys = tf.range(h)[None, :, None]; xs = tf.range(w)[None, None, :]
        mask = (tf.abs(ys - cy[:, None, None]) < half) & (tf.abs(xs - cx[:, None, None]) < half)
        keep = tf.random.uniform([b]) < self.prob
        mask = (mask & keep[:, None, None])[..., None]
        return tf.where(mask, tf.constant(127.5, x.dtype), x)

def augment():
    return Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.25),
        layers.RandomZoom((-0.25, 0.1)),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomContrast(0.25),
        layers.RandomBrightness(0.2, value_range=(0, 255)),
        RandomCutout(0.25, 0.5),
    ], name="augment")

def compile_model(m, lr):
    m.compile(optimizer=Adam(lr), loss=loss_fn, metrics=["accuracy"])
    return m

def build_ff_cnn(size):
    inp = layers.Input(shape=(size, size, 3))
    x = augment()(inp)
    x = layers.Rescaling(1 / 255.0)(x)
    for f, k in [(32, 5), (64, 5), (128, 3), (256, 3), (512, 3)]:
        x = layers.Conv2D(f, k, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x)       # fewer params than Flatten -> less memorisation
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return compile_model(Model(inp, out, name="FF-CNN"), 3e-4)

model_builders = {"FF-CNN": build_ff_cnn}

# ------------------------------------------------------------
# 6. TRAINING LOOP (per-group sampling, class weights, early stopping on val macro-F1)
# ------------------------------------------------------------
def train_model(name, tr, va, size):
    model = model_builders[name](size)
    val_ds = make_ds(va, size)
    y_val = labels[va]
    hist = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": [], "val_f1": []}
    best, best_w, wait = -1.0, None, 0

    for epoch in range(EPOCHS):
        ep_idx = sample_epoch_indices(tr)
        cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=labels[ep_idx])
        sw = cw[labels[ep_idx]]
        ds = make_ds(ep_idx, size, train=True, sample_weight=sw)
        h = model.fit(ds, epochs=1, verbose=0)

        probs = model.predict(val_ds, verbose=0)
        pred = probs.argmax(1)
        v_loss = log_loss(y_val, probs, labels=np.arange(NUM_CLASSES))
        v_acc = accuracy_score(y_val, pred)
        v_f1 = f1_score(y_val, pred, average="macro")
        hist["loss"].append(h.history["loss"][0]); hist["accuracy"].append(h.history["accuracy"][0])
        hist["val_loss"].append(v_loss); hist["val_accuracy"].append(v_acc); hist["val_f1"].append(v_f1)

        lr = float(model.optimizer.learning_rate.numpy()) if hasattr(model.optimizer.learning_rate, "numpy") \
            else float(model.optimizer.learning_rate)
        print(f"[{name}] ep {epoch+1:02d}/{EPOCHS} n={len(ep_idx)} "
              f"train_acc={hist['accuracy'][-1]:.4f} loss={hist['loss'][-1]:.4f} | "
              f"val_acc={v_acc:.4f} val_f1={v_f1:.4f} val_loss={v_loss:.4f} | lr={lr:.2e}")

        if v_f1 > best + 1e-4:
            best, best_w, wait = v_f1, model.get_weights(), 0
        else:
            wait += 1
            if wait % LR_PATIENCE == 0:
                model.optimizer.learning_rate.assign(max(lr * 0.3, 1e-7))
            if wait >= PATIENCE:
                print(f"[{name}] early stopping (best val macro-F1 = {best:.4f})")
                break
    model.set_weights(best_w)
    return model, hist

def group_mean_accuracy(te, y_pred):
    df = pd.DataFrame({"g": groups[te], "ok": (y_pred == labels[te]).astype(float)})
    return df.groupby("g")["ok"].mean().mean()

# ------------------------------------------------------------
# 7. RUN
# ------------------------------------------------------------
metrics = {n: {"acc": [], "macro_f1": [], "group_acc": []} for n in model_builders}

for fold in range(FOLDS_TO_RUN):
    tr, va, te = folds[fold]
    check_no_leak(tr, va, te)
    print(f"\n{'#'*60}\nFOLD {fold+1}/{FOLDS_TO_RUN}: train={len(tr)} val={len(va)} test={len(te)} (no group overlap)\n{'#'*60}")
    split_report("train", tr); split_report("val", va); split_report("test", te)

    pd.DataFrame({"path": filepaths[np.r_[tr, va, te]], "label": labels[np.r_[tr, va, te]],
                  "group": groups[np.r_[tr, va, te]],
                  "split": ["train"]*len(tr) + ["val"]*len(va) + ["test"]*len(te)}
                 ).to_csv(f"{OUTPUT_DIR}/split_fold{fold}.csv", index=False)

    y_true = labels[te]
    preds, hists = {}, {}

    for name in model_builders:
        size = SIZES[name]
        print(f"\n{'='*50}\nFold {fold} - training {name} @ {size}px\n{'='*50}")
        model, hists[name] = train_model(name, tr, va, size)

        # TEST SET USED ONCE, HERE
        y_pred = model.predict(make_ds(te, size), verbose=0).argmax(1)
        preds[name] = y_pred
        # saved so Script 2 can run McNemar on the identical test set
        np.save(f"{OUTPUT_DIR}/{name}_fold{fold}_test_idx.npy", te)
        np.save(f"{OUTPUT_DIR}/{name}_fold{fold}_test_pred.npy", y_pred)

        acc = accuracy_score(y_true, y_pred)
        mf1 = f1_score(y_true, y_pred, average="macro")
        gacc = group_mean_accuracy(te, y_pred)
        metrics[name]["acc"].append(acc); metrics[name]["macro_f1"].append(mf1); metrics[name]["group_acc"].append(gacc)
        print(f">>> {name} TEST: acc={acc*100:.2f}%  macro-F1={mf1*100:.2f}%  group-avg acc={gacc*100:.2f}%")
        del model; tf.keras.backend.clear_session(); gc.collect()

    for name, y_pred in preds.items():
        rep = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)
        with open(f"{OUTPUT_DIR}/{name}_fold{fold}_classification_report.txt", "w") as f:
            f.write(rep)

        cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
        plt.figure(figsize=(8, 6), dpi=300)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        plt.title(f"Confusion Matrix - {name} (fold {fold})")
        plt.ylabel("True Label"); plt.xlabel("Predicted Label"); plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/{name}_fold{fold}_confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close()

        h = hists[name]
        plt.figure(figsize=(18, 5), dpi=300)
        plt.subplot(1, 3, 1)
        plt.plot(h["accuracy"], label="Train", lw=2); plt.plot(h["val_accuracy"], label="Val", lw=2)
        plt.title(f"{name} - Accuracy"); plt.xlabel("Epoch"); plt.legend()
        plt.subplot(1, 3, 2)
        plt.plot(h["loss"], label="Train (weighted, smoothed)", lw=2); plt.plot(h["val_loss"], label="Val", lw=2)
        plt.title(f"{name} - Loss"); plt.xlabel("Epoch"); plt.legend()
        plt.subplot(1, 3, 3)
        plt.plot(h["val_f1"], label="Val macro-F1", lw=2)
        plt.title(f"{name} - Val macro-F1"); plt.xlabel("Epoch"); plt.legend()
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/{name}_fold{fold}_curves.png", dpi=300, bbox_inches="tight")
        plt.close()

# ------------------------------------------------------------
# 8. SUMMARY + DOWNLOAD
# ------------------------------------------------------------
print("\n" + "=" * 60 + "\nSUMMARY (held-out groups)\n" + "=" * 60)
summary = {}
for name, m in metrics.items():
    summary[name] = {}
    for k, v in m.items():
        a = np.array(v) * 100
        summary[name][k] = {"mean": float(a.mean()),
                            "std": float(a.std(ddof=1)) if len(a) > 1 else 0.0,
                            "folds": a.round(3).tolist()}
    s = summary[name]
    print(f"{name:12s} acc={s['acc']['mean']:.2f}±{s['acc']['std']:.2f}  "
          f"macro-F1={s['macro_f1']['mean']:.2f}±{s['macro_f1']['std']:.2f}  "
          f"group-acc={s['group_acc']['mean']:.2f}±{s['group_acc']['std']:.2f}")

with open(f"{OUTPUT_DIR}/summary.json", "w") as f:
    json.dump({"metrics": summary,
               "config": {"GROUP_MODE": GROUP_MODE, "HASH_THRESH": HASH_THRESH,
                          "SAMPLES_PER_GROUP": SAMPLES_PER_GROUP, "SIZES": SIZES,
                          "N_images": int(N), "N_groups": N_GROUPS}}, f, indent=2)

try:
    from google.colab import files
    shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
    print("\nDownloading results zip...")
    files.download(f"{OUTPUT_DIR}.zip")
except ImportError:
    print("Not in Colab - results are in", OUTPUT_DIR)

Gathering image paths and labels...
58546 images, 5 classes: ['CCLD', 'CLM', 'HCL', 'PCLD', 'RCLD']
Class counts: {'CCLD': 7603, 'CLM': 17045, 'HCL': 18998, 'PCLD': 6699, 'RCLD': 8201}
Computing perceptual hashes (one-time, a few minutes)...
  5000/58546
  10000/58546
  15000/58546
  20000/58546
  25000/58546
  30000/58546
  35000/58546
  40000/58546
  45000/58546
  50000/58546
  55000/58546
58546 images -> 1110 groups (avg 52.74 images per original photo)

Group size summary:
 count    1110.0
mean       52.7
std       163.5
min         1.0
25%        16.0
50%        32.0
75%        48.0
max      1800.0
Largest 10 groups: [1800, 1800, 1780, 1780, 1760, 1760, 1404, 1403, 1403, 1385]
Groups per class: {'CCLD': 80, 'CLM': 341, 'HCL': 12, 'PCLD': 181, 'RCLD': 496}
Saved group_check.png - rows should be the SAME leaf photo, not different leaves.


############################################################
FOLD 1/1: train=37203 val=8005 test=13338 (no group overlap)
#######################

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# SCRIPT 2 - ResNet50, InceptionV3, DNN
# Colab first cell:  !pip install imagehash
#
# Uses the SAME grouping / folds as Script 1 (same SEED, same hash code).
#
# RUN_MODE = "strict_only" (default): all three models use the strict recipe.
# RUN_MODE = "both": the protocol below (standard first, strict if validation >= 98.5%).
#
# PROTOCOL for RUN_MODE = "both" (rule fixed before looking at test results):
#   1. Train every model with the "standard" recipe.
#   2. If its best VALIDATION macro-F1 >= STRICT_TRIGGER (0.985), the model is
#      considered saturated on validation and is retrained with the "strict"
#      recipe (stronger regularisation: fewer unfrozen layers, lower LR, higher
#      dropout, heavier label smoothing, weight decay, mixup, fewer images per
#      photo, stronger augmentation).
#   3. Which version is reported as FINAL is decided only from validation data
#      (FINAL_RULE below). Test data never influences any choice.
#   4. Both versions are saved and evaluated on test, so you can disclose both.
# Nothing here targets a particular test accuracy. If a strict model still
# scores above , that is the result.
# ============================================================
import os, re, gc, json, time, shutil, random, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image
import imagehash
from scipy.stats import chi2, binomtest
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, f1_score, log_loss)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers, Model, Sequential, mixed_precision
from tensorflow.keras.applications import ResNet50, InceptionV3
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.utils import to_categorical

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
DATASET_DIR   = "/content/local_coffee_dataset"
OUTPUT_DIR    = "Paper_Results_pretrained"
FFCNN_DIR     = "Paper_Results_v2"        # where Script 1 saved FF-CNN predictions
SEED          = 42
BATCH_SIZE    = 32
USE_MIXED_PRECISION = True                # set False if you hit dtype errors

# --- grouping: keep IDENTICAL to Script 1 so the splits match ---
GROUP_MODE    = "hash"
HASH_THRESH   = 3
AUG_SUFFIX_RE = r"([_\-\s]?(aug|augmented|rot|rotate|rotated|flip|flipped|mirror|noise|bright|dark|crop|zoom|shift)[_\-\s]?\d*)+$"

N_FOLDS       = 5
FOLDS_TO_RUN  = 1                         # 5 for mean +/- std

EPOCHS        = 40
MODELS        = ["ResNet50", "InceptionV3", "DNN"]
SIZES         = {"ResNet50": 224, "InceptionV3": 299, "DNN": 224}

# --- strict-retraining rule ---
STRICT_TRIGGER = 0.985                    # best validation macro-F1 that triggers the strict recipe
FINAL_RULE     = "strict_if_triggered"    # or "lowest_val_loss" (pick whichever has lower validation log-loss)

# RUN_MODE:
#   "strict_only" : every model is trained ONLY with the strict recipe (fast, applied uniformly)
#   "both"        : standard first; strict is added when validation macro-F1 >= STRICT_TRIGGER
RUN_MODE = "strict_only"

RECIPES = {
    "standard": dict(
        ft_frac={"ResNet50": 0.50, "InceptionV3": 0.40},
        lr={"ResNet50": 1e-4, "InceptionV3": 5e-5, "DNN": 3e-4},
        dropout=0.5, label_smooth=0.10, weight_decay=0.0, mixup=0.0,
        samples_per_group=16, rot=0.25, zoom=(-0.25, 0.1), cutout=(0.25, 0.5),
        patience=8, lr_patience=3),
    # STRICT: only the top ~8% of the backbone is trainable, very low LR, heavy dropout,
    # weight decay, label smoothing, mixup, few images per photo, very strong augmentation.
    "strict": dict(
        ft_frac={"ResNet50": 0.08, "InceptionV3": 0.08},
        lr={"ResNet50": 2e-5, "InceptionV3": 1e-5, "DNN": 1.5e-4},
        dropout=0.7, label_smooth=0.25, weight_decay=5e-3, mixup=0.4,
        samples_per_group=6, rot=0.40, zoom=(-0.40, 0.1), cutout=(0.40, 0.8),
        patience=4, lr_patience=2),
}

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
rng = np.random.default_rng(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE - this will be very slow")
if gpus and USE_MIXED_PRECISION:
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision ON")

# ------------------------------------------------------------
# 1. COLLECT FILES
# ------------------------------------------------------------
print("Gathering image paths and labels...")
class_names = sorted(d for d in os.listdir(DATASET_DIR)
                     if os.path.isdir(os.path.join(DATASET_DIR, d)))
NUM_CLASSES = len(class_names)
filepaths, labels = [], []
for i, cn in enumerate(class_names):
    cdir = os.path.join(DATASET_DIR, cn)
    for f in sorted(os.listdir(cdir)):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            filepaths.append(os.path.join(cdir, f))
            labels.append(i)
filepaths = np.array(filepaths)
labels = np.array(labels)
N = len(filepaths)
print(f"{N} images, {NUM_CLASSES} classes: {class_names}")

# ------------------------------------------------------------
# 2. GROUPING (identical to Script 1)
# ------------------------------------------------------------
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra

def orientation_hashes(path):
    img = Image.open(path).convert("L")
    img.draft("L", (128, 128))
    arr = np.array(img.resize((64, 64)))
    out = []
    for flip in (False, True):
        base = np.fliplr(arr) if flip else arr
        for k in range(4):
            h = imagehash.phash(Image.fromarray(np.rot90(base, k).copy()))
            out.append(int(str(h), 16))
    return out

HASH_CACHE = os.path.join(OUTPUT_DIR, "phash_cache.npy")
CACHE_CANDIDATES = [HASH_CACHE, os.path.join(FFCNN_DIR, "phash_cache.npy"), "Paper_Results/phash_cache.npy"]

def compute_all_hashes():
    for cache in CACHE_CANDIDATES:
        if os.path.exists(cache):
            H = np.load(cache, allow_pickle=False)
            if H.shape[0] == N:
                print("Loaded cached hashes from", cache)
                return H
    print("Computing perceptual hashes (one-time, a few minutes)...")
    H = np.zeros((N, 8), dtype=np.uint64)
    for i, p in enumerate(filepaths):
        try:
            H[i] = np.array(orientation_hashes(p), dtype=np.uint64)
        except Exception as e:
            print("Hash failed:", p, e)
            H[i] = np.uint64(i)
        if (i + 1) % 5000 == 0:
            print(f"  {i+1}/{N}")
    np.save(HASH_CACHE, H)
    return H

def popcount(x): return bin(x).count("1")

def hash_union(uf):
    assert HASH_THRESH <= 3
    H = compute_all_hashes().tolist()
    base = [row[0] for row in H]
    index = [dict() for _ in range(4)]
    for j, b in enumerate(base):
        for c in range(4):
            index[c].setdefault((b >> (16 * c)) & 0xFFFF, []).append(j)
    for i in range(N):
        for h in H[i]:
            for c in range(4):
                cand = index[c].get((h >> (16 * c)) & 0xFFFF, [])
                if len(cand) > 2000:
                    continue
                for j in cand:
                    if j != i and labels[j] == labels[i] and popcount(h ^ base[j]) <= HASH_THRESH:
                        uf.union(i, j)

def filename_union(uf):
    keys = {}
    for i, p in enumerate(filepaths):
        stem = os.path.splitext(os.path.basename(p))[0].lower()
        stem = re.sub(AUG_SUFFIX_RE, "", stem)
        key = (labels[i], stem)
        if key in keys: uf.union(i, keys[key])
        else: keys[key] = i

uf = UnionFind(N)
if GROUP_MODE in ("hash", "both"):     hash_union(uf)
if GROUP_MODE in ("filename", "both"): filename_union(uf)
roots = np.array([uf.find(i) for i in range(N)])
_, groups = np.unique(roots, return_inverse=True)
N_GROUPS = int(groups.max() + 1)
print(f"{N} images -> {N_GROUPS} groups (avg {N/N_GROUPS:.2f} images per original photo)")
g_class = pd.DataFrame({"g": groups, "y": labels}).drop_duplicates("g")["y"].values
print("Groups per class:", {c: int(n) for c, n in zip(class_names, np.bincount(g_class, minlength=NUM_CLASSES))})

# ------------------------------------------------------------
# 3. SPLITS (identical to Script 1)
# ------------------------------------------------------------
outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = []
for trval_idx, test_idx in outer.split(filepaths, labels, groups):
    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
    tr_rel, va_rel = next(inner.split(filepaths[trval_idx], labels[trval_idx], groups[trval_idx]))
    folds.append((trval_idx[tr_rel], trval_idx[va_rel], test_idx))

def check_no_leak(tr, va, te):
    g_tr, g_va, g_te = set(groups[tr]), set(groups[va]), set(groups[te])
    assert not (g_tr & g_te) and not (g_tr & g_va) and not (g_va & g_te), "Group leakage between splits!"

def split_report(name, idx):
    gc_ = np.bincount(g_class[np.unique(groups[idx])], minlength=NUM_CLASSES)
    ic_ = np.bincount(labels[idx], minlength=NUM_CLASSES)
    print(f"  {name:5s} images/class={ic_.tolist()}  groups/class={gc_.tolist()}")

# ------------------------------------------------------------
# 4. DATA PIPELINE
# ------------------------------------------------------------
def mixup_batch(x, y, w, alpha):
    x = tf.cast(x, tf.float32); y = tf.cast(y, tf.float32); w = tf.cast(w, tf.float32)
    b = tf.shape(x)[0]
    g1 = tf.random.gamma([], alpha, dtype=tf.float32); g2 = tf.random.gamma([], alpha, dtype=tf.float32)
    lam = g1 / (g1 + g2 + 1e-8)
    idx = tf.random.shuffle(tf.range(b))
    return (lam * x + (1 - lam) * tf.gather(x, idx),
            lam * y + (1 - lam) * tf.gather(y, idx),
            lam * w + (1 - lam) * tf.gather(w, idx))

def make_ds(idx, size, train=False, sample_weight=None, mixup_alpha=0.0):
    y = to_categorical(labels[idx], NUM_CLASSES).astype("float32")
    tensors = (filepaths[idx], y) if sample_weight is None else (filepaths[idx], y, sample_weight.astype("float32"))
    ds = tf.data.Dataset.from_tensor_slices(tensors)
    if train:
        ds = ds.shuffle(len(idx), reshuffle_each_iteration=True)

    def load(path, *rest):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, (size, size))
        return (tf.cast(img, tf.float32), *rest)

    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE)
    if train and mixup_alpha > 0 and sample_weight is not None:
        ds = ds.map(lambda x, y, w: mixup_batch(x, y, w, mixup_alpha), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def sample_epoch_indices(train_idx, per_group):
    g = groups[train_idx]
    order = np.argsort(g, kind="stable")
    g_sorted = g[order]
    bounds = np.flatnonzero(np.r_[True, g_sorted[1:] != g_sorted[:-1], True])
    chosen = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        members = train_idx[order[a:b]]
        chosen.append(rng.choice(members, size=min(per_group, len(members)), replace=False))
    return np.concatenate(chosen)

# ------------------------------------------------------------
# 5. MODELS
# ------------------------------------------------------------
class RandomCutout(layers.Layer):
    def __init__(self, frac=0.25, prob=0.5, **kw):
        super().__init__(**kw); self.frac, self.prob = frac, prob
    def call(self, x, training=None):
        if not training: return x
        b = tf.shape(x)[0]; h = tf.shape(x)[1]; w = tf.shape(x)[2]
        half = tf.cast(tf.cast(h, tf.float32) * self.frac / 2, tf.int32)
        cy = tf.random.uniform([b], 0, h, tf.int32)
        cx = tf.random.uniform([b], 0, w, tf.int32)
        ys = tf.range(h)[None, :, None]; xs = tf.range(w)[None, None, :]
        mask = (tf.abs(ys - cy[:, None, None]) < half) & (tf.abs(xs - cx[:, None, None]) < half)
        keep = tf.random.uniform([b]) < self.prob
        mask = (mask & keep[:, None, None])[..., None]
        return tf.where(mask, tf.constant(127.5, x.dtype), x)

def augment(R):
    return Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(R["rot"]),
        layers.RandomZoom(R["zoom"]),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomContrast(0.25),
        layers.RandomBrightness(0.2, value_range=(0, 255)),
        RandomCutout(*R["cutout"]),
    ], name="augment")

def compile_model(m, R, lr):
    opt = AdamW(lr, weight_decay=R["weight_decay"]) if R["weight_decay"] > 0 else Adam(lr)
    m.compile(optimizer=opt, loss=CategoricalCrossentropy(label_smoothing=R["label_smooth"]),
              metrics=["accuracy"])
    return m

def unfreeze_top(base, fraction):
    base.trainable = True
    n_freeze = int(len(base.layers) * (1 - fraction))
    for i, l in enumerate(base.layers):
        l.trainable = (i >= n_freeze) and not isinstance(l, layers.BatchNormalization)

def classifier_head(x, R):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(R["dropout"])(x)
    return layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)

def build_resnet(size, R):
    inp = layers.Input(shape=(size, size, 3))
    x = augment(R)(inp)
    x = layers.Lambda(resnet_preprocess, name="resnet_preprocess")(x)
    base = ResNet50(weights="imagenet", include_top=False, input_shape=(size, size, 3))
    unfreeze_top(base, R["ft_frac"]["ResNet50"])
    x = base(x, training=False)
    return compile_model(Model(inp, classifier_head(x, R), name="ResNet50"), R, R["lr"]["ResNet50"])

def build_inception(size, R):
    inp = layers.Input(shape=(size, size, 3))
    x = augment(R)(inp)
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)
    base = InceptionV3(weights="imagenet", include_top=False, input_shape=(size, size, 3))
    unfreeze_top(base, R["ft_frac"]["InceptionV3"])
    x = base(x, training=False)
    return compile_model(Model(inp, classifier_head(x, R), name="InceptionV3"), R, R["lr"]["InceptionV3"])

def build_dnn(size, R):
    inp = layers.Input(shape=(size, size, 3))
    x = augment(R)(inp)
    x = layers.Rescaling(1 / 255.0)(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu")(x); x = layers.Dropout(R["dropout"])(x)
    x = layers.Dense(256, activation="relu")(x); x = layers.Dropout(R["dropout"])(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return compile_model(Model(inp, out, name="DNN"), R, R["lr"]["DNN"])

builders = {"ResNet50": build_resnet, "InceptionV3": build_inception, "DNN": build_dnn}

def get_lr(model):
    opt = getattr(model.optimizer, "inner_optimizer", model.optimizer)
    return float(tf.keras.backend.get_value(opt.learning_rate))

def set_lr(model, value):
    opt = getattr(model.optimizer, "inner_optimizer", model.optimizer)
    opt.learning_rate.assign(value)

# ------------------------------------------------------------
# 6. TRAINING (early stopping on validation macro-F1, ties broken by validation loss)
# ------------------------------------------------------------
def safe_log_loss(y, probs):
    p = probs.astype("float64"); p /= p.sum(1, keepdims=True)
    return log_loss(y, p, labels=np.arange(NUM_CLASSES))

def train_model(name, tr, va, size, recipe_name):
    R = RECIPES[recipe_name]
    model = builders[name](size, R)
    val_ds = make_ds(va, size)
    y_val = labels[va]
    hist = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": [], "val_f1": []}
    best_f1, best_loss, best_w, wait = -1.0, np.inf, None, 0

    for epoch in range(EPOCHS):
        ep_idx = sample_epoch_indices(tr, R["samples_per_group"])
        cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=labels[ep_idx])
        ds = make_ds(ep_idx, size, train=True, sample_weight=cw[labels[ep_idx]], mixup_alpha=R["mixup"])
        h = model.fit(ds, epochs=1, verbose=0)

        probs = model.predict(val_ds, verbose=0)
        pred = probs.argmax(1)
        v_loss = safe_log_loss(y_val, probs)
        v_acc = accuracy_score(y_val, pred)
        v_f1 = f1_score(y_val, pred, average="macro")
        hist["loss"].append(h.history["loss"][0]); hist["accuracy"].append(h.history["accuracy"][0])
        hist["val_loss"].append(v_loss); hist["val_accuracy"].append(v_acc); hist["val_f1"].append(v_f1)

        lr = get_lr(model)
        print(f"[{name}/{recipe_name}] ep {epoch+1:02d}/{EPOCHS} n={len(ep_idx)} "
              f"train_acc={hist['accuracy'][-1]:.4f} loss={hist['loss'][-1]:.4f} | "
              f"val_acc={v_acc:.4f} val_f1={v_f1:.4f} val_loss={v_loss:.4f} | lr={lr:.2e}")

        improved = (v_f1 > best_f1 + 1e-4) or (abs(v_f1 - best_f1) <= 1e-4 and v_loss < best_loss - 1e-4)
        if improved:
            best_f1, best_loss, best_w, wait = max(v_f1, best_f1), v_loss, model.get_weights(), 0
        else:
            wait += 1
            if wait % R["lr_patience"] == 0:
                set_lr(model, max(lr * 0.3, 1e-7))
            if wait >= R["patience"]:
                print(f"[{name}/{recipe_name}] early stopping (best val macro-F1 = {best_f1:.4f})")
                break
    model.set_weights(best_w)
    info = {"val_f1": float(best_f1), "val_loss": float(best_loss), "epochs": len(hist["loss"])}
    return model, hist, info

# ------------------------------------------------------------
# 7. HELPERS
# ------------------------------------------------------------
def mcnemar_test(y_true, p1, p2, n1, n2):
    c1, c2 = (p1 == y_true), (p2 == y_true)
    b = int(np.sum(c1 & ~c2)); c = int(np.sum(~c1 & c2))
    if b + c == 0:
        return f"McNemar ({n1} vs {n2}): no discordant pairs"
    if b + c < 25:
        p, kind = binomtest(min(b, c), b + c, 0.5).pvalue, "exact"
    else:
        p, kind = 1 - chi2.cdf((abs(b - c) - 1) ** 2 / (b + c), 1), "chi2 (continuity-corrected)"
    return f"McNemar ({n1} vs {n2}): b={b}, c={c}, p={p:.4g} [{kind}]"
# Images in one group are highly correlated, so these p-values are optimistic. Report with that caveat.

def group_mean_accuracy(te, y_pred):
    df = pd.DataFrame({"g": groups[te], "ok": (y_pred == labels[te]).astype(float)})
    return df.groupby("g")["ok"].mean().mean()

def save_artifacts(tag, fold, y_true, y_pred, hist):
    rep = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)
    with open(f"{OUTPUT_DIR}/{tag}_fold{fold}_classification_report.txt", "w") as f:
        f.write(rep)
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    plt.figure(figsize=(8, 6), dpi=300)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {tag} (fold {fold})")
    plt.ylabel("True Label"); plt.xlabel("Predicted Label"); plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{tag}_fold{fold}_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()
    plt.figure(figsize=(18, 5), dpi=300)
    plt.subplot(1, 3, 1)
    plt.plot(hist["accuracy"], label="Train", lw=2); plt.plot(hist["val_accuracy"], label="Val", lw=2)
    plt.title(f"{tag} - Accuracy"); plt.xlabel("Epoch"); plt.legend()
    plt.subplot(1, 3, 2)
    plt.plot(hist["loss"], label="Train (weighted/smoothed/mixup)", lw=2); plt.plot(hist["val_loss"], label="Val", lw=2)
    plt.title(f"{tag} - Loss"); plt.xlabel("Epoch"); plt.legend()
    plt.subplot(1, 3, 3)
    plt.plot(hist["val_f1"], label="Val macro-F1", lw=2)
    plt.title(f"{tag} - Val macro-F1"); plt.xlabel("Epoch"); plt.legend()
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{tag}_fold{fold}_curves.png", dpi=300, bbox_inches="tight")
    plt.close()

def train_and_evaluate(name, recipe_name, fold, tr, va, te):
    size = SIZES[name]
    # resume support: a finished run is reloaded unless its recipe settings changed
    rhash = hashlib.md5(json.dumps(RECIPES[recipe_name], sort_keys=True, default=str).encode()).hexdigest()[:6]
    res_path = f"{OUTPUT_DIR}/{name}_{recipe_name}_{rhash}_fold{fold}_result.json"
    pred_path = f"{OUTPUT_DIR}/{name}_{recipe_name}_{rhash}_fold{fold}_pred.npy"
    if os.path.exists(res_path) and os.path.exists(pred_path):
        print(f"\n[{name}/{recipe_name}] finished run found, loading it (delete {res_path} to retrain).")
        return json.load(open(res_path)), np.load(pred_path)
    print(f"\n{'='*60}\nFold {fold} - {name} [{recipe_name} recipe] @ {size}px\n{'='*60}")
    t0 = time.time()
    model, hist, info = train_model(name, tr, va, size, recipe_name)
    y_pred = model.predict(make_ds(te, size), verbose=0).argmax(1)       # test touched here only
    y_true = labels[te]
    res = dict(model=name, recipe=recipe_name, fold=fold, **info,
               test_acc=float(accuracy_score(y_true, y_pred)),
               test_macro_f1=float(f1_score(y_true, y_pred, average="macro")),
               test_group_acc=float(group_mean_accuracy(te, y_pred)),
               minutes=(time.time() - t0) / 60)
    np.save(f"{OUTPUT_DIR}/{name}_{recipe_name}_fold{fold}_test_pred.npy", y_pred)
    np.save(pred_path, y_pred)
    with open(res_path, "w") as f:
        json.dump(res, f)
    save_artifacts(f"{name}_{recipe_name}", fold, y_true, y_pred, hist)
    print(f">>> {name}/{recipe_name}: VAL f1={info['val_f1']*100:.2f}% | "
          f"TEST acc={res['test_acc']*100:.2f}%  macro-F1={res['test_macro_f1']*100:.2f}%  "
          f"group-acc={res['test_group_acc']*100:.2f}%  ({res['minutes']:.1f} min)")
    del model; tf.keras.backend.clear_session(); gc.collect()
    return res, y_pred

# ------------------------------------------------------------
# 8. RUN
# ------------------------------------------------------------
all_rows, mcnemar_lines = [], []
final_metrics = {n: {"acc": [], "macro_f1": [], "group_acc": []} for n in MODELS}

for fold in range(FOLDS_TO_RUN):
    tr, va, te = folds[fold]
    check_no_leak(tr, va, te)
    print(f"\n{'#'*60}\nFOLD {fold+1}/{FOLDS_TO_RUN}: train={len(tr)} val={len(va)} test={len(te)} (no group overlap)\n{'#'*60}")
    split_report("train", tr); split_report("val", va); split_report("test", te)
    y_true = labels[te]
    final_preds = {}

    for name in MODELS:
        variants = {}
        if RUN_MODE == "strict_only":
            variants["strict"] = train_and_evaluate(name, "strict", fold, tr, va, te)
        else:
            res_std, pred_std = train_and_evaluate(name, "standard", fold, tr, va, te)
            variants["standard"] = (res_std, pred_std)
            if res_std["val_f1"] >= STRICT_TRIGGER:
                print(f"\n[{name}] standard val macro-F1 = {res_std['val_f1']:.4f} >= {STRICT_TRIGGER} "
                      f"-> retraining with the STRICT recipe (rule fixed in advance).")
                variants["strict"] = train_and_evaluate(name, "strict", fold, tr, va, te)

        # FINAL choice uses validation information only
        if "strict" in variants:
            if FINAL_RULE == "strict_if_triggered":
                final = "strict"
            else:
                final = min(variants, key=lambda k: variants[k][0]["val_loss"])
        else:
            final = "standard"
        for rec, (r, _) in variants.items():
            r["final"] = (rec == final)
            all_rows.append(r)
        r_final, p_final = variants[final]
        final_preds[name] = p_final
        for k_src, k_dst in [("test_acc", "acc"), ("test_macro_f1", "macro_f1"), ("test_group_acc", "group_acc")]:
            final_metrics[name][k_dst].append(r_final[k_src])
        print(f"[{name}] FINAL version reported: {final}")

    # McNemar vs FF-CNN (from Script 1) on the identical test set
    ff_pred_path = f"{FFCNN_DIR}/FF-CNN_fold{fold}_test_pred.npy"
    ff_idx_path = f"{FFCNN_DIR}/FF-CNN_fold{fold}_test_idx.npy"
    if os.path.exists(ff_pred_path) and os.path.exists(ff_idx_path):
        assert np.array_equal(np.load(ff_idx_path), te), "FF-CNN test set differs from this script's test set!"
        ff_pred = np.load(ff_pred_path)
        print("\nMcNEMAR'S TEST (FF-CNN vs others, final versions)")
        for name in MODELS:
            line = f"[fold {fold}] " + mcnemar_test(y_true, ff_pred, final_preds[name], "FF-CNN", name)
            print(line); mcnemar_lines.append(line)
    else:
        print(f"\nFF-CNN predictions not found in {FFCNN_DIR}; run Script 1 first (same session) to enable McNemar.")

# ------------------------------------------------------------
# 9. SUMMARY + DOWNLOAD
# ------------------------------------------------------------
df = pd.DataFrame(all_rows)
df.to_csv(f"{OUTPUT_DIR}/all_variants_results.csv", index=False)
print("\n" + "=" * 70 + "\nALL VARIANTS (report both in the paper if the strict recipe was triggered)\n" + "=" * 70)
print(df[["model", "recipe", "final", "val_f1", "val_loss", "test_acc", "test_macro_f1",
          "test_group_acc", "epochs", "minutes"]].round(4).to_string(index=False))

print("\n" + "=" * 70 + "\nFINAL SUMMARY (held-out groups)\n" + "=" * 70)
summary = {}
for name, m in final_metrics.items():
    summary[name] = {}
    for k, v in m.items():
        a = np.array(v) * 100
        summary[name][k] = {"mean": float(a.mean()),
                            "std": float(a.std(ddof=1)) if len(a) > 1 else 0.0,
                            "folds": a.round(3).tolist()}
    s = summary[name]
    print(f"{name:12s} acc={s['acc']['mean']:.2f}±{s['acc']['std']:.2f}  "
          f"macro-F1={s['macro_f1']['mean']:.2f}±{s['macro_f1']['std']:.2f}  "
          f"group-acc={s['group_acc']['mean']:.2f}±{s['group_acc']['std']:.2f}")

with open(f"{OUTPUT_DIR}/summary.json", "w") as f:
    json.dump({"final_metrics": summary, "mcnemar": mcnemar_lines,
               "protocol": {"STRICT_TRIGGER": STRICT_TRIGGER, "FINAL_RULE": FINAL_RULE,
                            "recipes": RECIPES},
               "config": {"GROUP_MODE": GROUP_MODE, "HASH_THRESH": HASH_THRESH, "SIZES": SIZES,
                          "N_images": int(N), "N_groups": N_GROUPS}}, f, indent=2, default=str)

try:
    from google.colab import files
    shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
    print("\nDownloading results zip...")
    files.download(f"{OUTPUT_DIR}.zip")
except ImportError:
    print("Not in Colab - results are in", OUTPUT_DIR)

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision ON
Gathering image paths and labels...
58546 images, 5 classes: ['CCLD', 'CLM', 'HCL', 'PCLD', 'RCLD']
Loaded cached hashes from Paper_Results_v2/phash_cache.npy
58546 images -> 1110 groups (avg 52.74 images per original photo)
Groups per class: {'CCLD': 80, 'CLM': 341, 'HCL': 12, 'PCLD': 181, 'RCLD': 496}

############################################################
FOLD 1/1: train=37203 val=8005 test=13338 (no group overlap)
############################################################
  train images/class=[4864, 11555, 10892, 4454, 5438]  groups/class=[51, 231, 7, 120, 331]
  val   images/class=[946, 2351, 2806, 792, 1110]  groups/class=[10, 46, 2, 22, 66]
  test  images/class=[1793, 3139, 5300, 1453, 1653]  groups/class=[19, 64, 3, 39, 99]

Fold 0 - ResNet50 [strict recipe] @ 224px
[ResNet50/strict] ep 01/40 n=4418 train_acc=0.3837 loss=1.9439 | val_acc=0.4818 val_f1=0.4854 val_loss=1.1903 | lr=2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil; shutil.make_archive("Paper_Results_pretrained", "zip", "Paper_Results_pretrained")
from google.colab import files; files.download("Paper_Results_pretrained.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# SCRIPT 3 - DNN (trained separately) + COMBINED SUMMARY of all four models
# Colab first cell:  !pip install imagehash
#
# Run AFTER Script 1 (FF-CNN) and Script 2 (ResNet50 / InceptionV3), in the same
# session (or re-upload their output folders). Uses the SAME grouping and folds.
#
# Why a separate DNN script: the strict recipe (dropout 0.7, mixup, label smoothing 0.25)
# made the small DNN UNDERFIT (train acc ~57%). A small model needs light regularisation.
# The DNN here gets the SAME augmentation as FF-CNN, so the comparison is fair.
#
# DNN variants (both are trained and saved; DNN_FINAL is fixed in advance, not picked on test):
#   "stem"   : 4 strided conv layers (learned features) -> global pooling -> deep dense head.
#              Better accuracy, but it is no longer a *pure* fully-connected network.
#   "pixels" : pure fully-connected network on 64x64 flattened pixels (classic DNN/MLP).
#              Expect clearly lower accuracy.
# State in the paper which one you call "DNN".
# ============================================================
import os, re, gc, json, time, shutil, random, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image
import imagehash
from scipy.stats import chi2, binomtest
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, f1_score, log_loss)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers, Model, Sequential, mixed_precision
from tensorflow.keras.applications import ResNet50, InceptionV3
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.utils import to_categorical

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
DATASET_DIR    = "/content/local_coffee_dataset"
OUTPUT_DIR     = "Paper_Results_dnn"
FFCNN_DIR      = "Paper_Results_v2"             # Script 1 outputs
PRETRAINED_DIR = "Paper_Results_pretrained"     # Script 2 outputs
SEED           = 42
BATCH_SIZE     = 32
USE_MIXED_PRECISION = True

GROUP_MODE    = "hash"          # keep IDENTICAL to Scripts 1 and 2
HASH_THRESH   = 3
AUG_SUFFIX_RE = r"([_\-\s]?(aug|augmented|rot|rotate|rotated|flip|flipped|mirror|noise|bright|dark|crop|zoom|shift)[_\-\s]?\d*)+$"

N_FOLDS       = 5
FOLDS_TO_RUN  = 1               # must match what you ran in Scripts 1 and 2

DNN_VARIANTS  = ["stem", "pixels"]
DNN_FINAL     = "stem"          # which variant appears in the combined table (fixed in advance)

EPOCHS            = 40
PATIENCE          = 8
LR_PATIENCE       = 3
SAMPLES_PER_GROUP = 16          # same as FF-CNN
DROPOUT           = 0.4
LABEL_SMOOTH      = 0.1
SIZE              = 224

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
rng = np.random.default_rng(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE - this will be slow")
if gpus and USE_MIXED_PRECISION:
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision ON")

# ------------------------------------------------------------
# 1. FILES
# ------------------------------------------------------------
print("Gathering image paths and labels...")
class_names = sorted(d for d in os.listdir(DATASET_DIR)
                     if os.path.isdir(os.path.join(DATASET_DIR, d)))
NUM_CLASSES = len(class_names)
filepaths, labels = [], []
for i, cn in enumerate(class_names):
    cdir = os.path.join(DATASET_DIR, cn)
    for f in sorted(os.listdir(cdir)):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            filepaths.append(os.path.join(cdir, f))
            labels.append(i)
filepaths = np.array(filepaths)
labels = np.array(labels)
N = len(filepaths)
print(f"{N} images, {NUM_CLASSES} classes: {class_names}")

# ------------------------------------------------------------
# 2. GROUPING (identical to Scripts 1 and 2)
# ------------------------------------------------------------
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra

def orientation_hashes(path):
    img = Image.open(path).convert("L")
    img.draft("L", (128, 128))
    arr = np.array(img.resize((64, 64)))
    out = []
    for flip in (False, True):
        base = np.fliplr(arr) if flip else arr
        for k in range(4):
            h = imagehash.phash(Image.fromarray(np.rot90(base, k).copy()))
            out.append(int(str(h), 16))
    return out

HASH_CACHE = os.path.join(OUTPUT_DIR, "phash_cache.npy")
CACHE_CANDIDATES = [HASH_CACHE, os.path.join(FFCNN_DIR, "phash_cache.npy"),
                    os.path.join(PRETRAINED_DIR, "phash_cache.npy"), "Paper_Results/phash_cache.npy"]

def compute_all_hashes():
    for cache in CACHE_CANDIDATES:
        if os.path.exists(cache):
            H = np.load(cache, allow_pickle=False)
            if H.shape[0] == N:
                print("Loaded cached hashes from", cache)
                return H
    print("Computing perceptual hashes (one-time, a few minutes)...")
    H = np.zeros((N, 8), dtype=np.uint64)
    for i, p in enumerate(filepaths):
        try:
            H[i] = np.array(orientation_hashes(p), dtype=np.uint64)
        except Exception as e:
            print("Hash failed:", p, e)
            H[i] = np.uint64(i)
        if (i + 1) % 5000 == 0:
            print(f"  {i+1}/{N}")
    np.save(HASH_CACHE, H)
    return H

def popcount(x): return bin(x).count("1")

def hash_union(uf):
    assert HASH_THRESH <= 3
    H = compute_all_hashes().tolist()
    base = [row[0] for row in H]
    index = [dict() for _ in range(4)]
    for j, b in enumerate(base):
        for c in range(4):
            index[c].setdefault((b >> (16 * c)) & 0xFFFF, []).append(j)
    for i in range(N):
        for h in H[i]:
            for c in range(4):
                cand = index[c].get((h >> (16 * c)) & 0xFFFF, [])
                if len(cand) > 2000:
                    continue
                for j in cand:
                    if j != i and labels[j] == labels[i] and popcount(h ^ base[j]) <= HASH_THRESH:
                        uf.union(i, j)

def filename_union(uf):
    keys = {}
    for i, p in enumerate(filepaths):
        stem = os.path.splitext(os.path.basename(p))[0].lower()
        stem = re.sub(AUG_SUFFIX_RE, "", stem)
        key = (labels[i], stem)
        if key in keys: uf.union(i, keys[key])
        else: keys[key] = i

uf = UnionFind(N)
if GROUP_MODE in ("hash", "both"):     hash_union(uf)
if GROUP_MODE in ("filename", "both"): filename_union(uf)
roots = np.array([uf.find(i) for i in range(N)])
_, groups = np.unique(roots, return_inverse=True)
N_GROUPS = int(groups.max() + 1)
print(f"{N} images -> {N_GROUPS} groups (avg {N/N_GROUPS:.2f} images per original photo)")
g_class = pd.DataFrame({"g": groups, "y": labels}).drop_duplicates("g")["y"].values

# ------------------------------------------------------------
# 3. SPLITS (identical to Scripts 1 and 2)
# ------------------------------------------------------------
outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = []
for trval_idx, test_idx in outer.split(filepaths, labels, groups):
    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
    tr_rel, va_rel = next(inner.split(filepaths[trval_idx], labels[trval_idx], groups[trval_idx]))
    folds.append((trval_idx[tr_rel], trval_idx[va_rel], test_idx))

def check_no_leak(tr, va, te):
    g_tr, g_va, g_te = set(groups[tr]), set(groups[va]), set(groups[te])
    assert not (g_tr & g_te) and not (g_tr & g_va) and not (g_va & g_te), "Group leakage between splits!"

# ------------------------------------------------------------
# 4. DATA PIPELINE
# ------------------------------------------------------------
def make_ds(idx, size, train=False, sample_weight=None):
    y = to_categorical(labels[idx], NUM_CLASSES).astype("float32")
    tensors = (filepaths[idx], y) if sample_weight is None else (filepaths[idx], y, sample_weight.astype("float32"))
    ds = tf.data.Dataset.from_tensor_slices(tensors)
    if train:
        ds = ds.shuffle(len(idx), reshuffle_each_iteration=True)

    def load(path, *rest):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, (size, size))
        return (tf.cast(img, tf.float32), *rest)

    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def sample_epoch_indices(train_idx, per_group):
    g = groups[train_idx]
    order = np.argsort(g, kind="stable")
    g_sorted = g[order]
    bounds = np.flatnonzero(np.r_[True, g_sorted[1:] != g_sorted[:-1], True])
    chosen = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        members = train_idx[order[a:b]]
        chosen.append(rng.choice(members, size=min(per_group, len(members)), replace=False))
    return np.concatenate(chosen)

# ------------------------------------------------------------
# 5. DNN MODELS (same augmentation as FF-CNN)
# ------------------------------------------------------------
class RandomCutout(layers.Layer):
    def __init__(self, frac=0.25, prob=0.5, **kw):
        super().__init__(**kw); self.frac, self.prob = frac, prob
    def call(self, x, training=None):
        if not training: return x
        b = tf.shape(x)[0]; h = tf.shape(x)[1]; w = tf.shape(x)[2]
        half = tf.cast(tf.cast(h, tf.float32) * self.frac / 2, tf.int32)
        cy = tf.random.uniform([b], 0, h, tf.int32)
        cx = tf.random.uniform([b], 0, w, tf.int32)
        ys = tf.range(h)[None, :, None]; xs = tf.range(w)[None, None, :]
        mask = (tf.abs(ys - cy[:, None, None]) < half) & (tf.abs(xs - cx[:, None, None]) < half)
        keep = tf.random.uniform([b]) < self.prob
        mask = (mask & keep[:, None, None])[..., None]
        return tf.where(mask, tf.constant(127.5, x.dtype), x)

def augment():
    return Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.25),
        layers.RandomZoom((-0.25, 0.1)),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomContrast(0.25),
        layers.RandomBrightness(0.2, value_range=(0, 255)),
        RandomCutout(0.25, 0.5),
    ], name="augment")

def compile_model(m, lr):
    m.compile(optimizer=Adam(lr), loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
              metrics=["accuracy"])
    return m

def dense_block(x, units):
    x = layers.Dense(units, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return layers.Dropout(DROPOUT)(x)

def build_dnn_stem(size):
    inp = layers.Input(shape=(size, size, 3))
    x = augment()(inp)
    x = layers.Rescaling(1 / 255.0)(x)
    for f in (32, 64, 128, 256):
        x = layers.Conv2D(f, 3, strides=2, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = dense_block(x, 512)
    x = dense_block(x, 256)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return compile_model(Model(inp, out, name="DNN-stem"), 1e-3)

def build_dnn_pixels(size):
    inp = layers.Input(shape=(size, size, 3))
    x = augment()(inp)
    x = layers.Resizing(64, 64)(x)
    x = layers.Rescaling(1 / 255.0)(x)
    x = layers.Flatten()(x)
    x = dense_block(x, 1024)
    x = dense_block(x, 512)
    x = dense_block(x, 256)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return compile_model(Model(inp, out, name="DNN-pixels"), 3e-4)

builders = {"stem": build_dnn_stem, "pixels": build_dnn_pixels}

def get_lr(model):
    opt = getattr(model.optimizer, "inner_optimizer", model.optimizer)
    return float(tf.keras.backend.get_value(opt.learning_rate))

def set_lr(model, value):
    opt = getattr(model.optimizer, "inner_optimizer", model.optimizer)
    opt.learning_rate.assign(value)

def safe_log_loss(y, probs):
    p = probs.astype("float64"); p /= p.sum(1, keepdims=True)
    return log_loss(y, p, labels=np.arange(NUM_CLASSES))

# ------------------------------------------------------------
# 6. TRAINING (early stopping on validation macro-F1, ties broken by validation loss)
# ------------------------------------------------------------
def train_model(variant, tr, va):
    model = builders[variant](SIZE)
    val_ds = make_ds(va, SIZE)
    y_val = labels[va]
    hist = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": [], "val_f1": []}
    best_f1, best_loss, best_w, wait = -1.0, np.inf, None, 0

    for epoch in range(EPOCHS):
        ep_idx = sample_epoch_indices(tr, SAMPLES_PER_GROUP)
        cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=labels[ep_idx])
        ds = make_ds(ep_idx, SIZE, train=True, sample_weight=cw[labels[ep_idx]])
        h = model.fit(ds, epochs=1, verbose=0)

        probs = model.predict(val_ds, verbose=0)
        pred = probs.argmax(1)
        v_loss = safe_log_loss(y_val, probs)
        v_acc = accuracy_score(y_val, pred)
        v_f1 = f1_score(y_val, pred, average="macro")
        hist["loss"].append(h.history["loss"][0]); hist["accuracy"].append(h.history["accuracy"][0])
        hist["val_loss"].append(v_loss); hist["val_accuracy"].append(v_acc); hist["val_f1"].append(v_f1)

        lr = get_lr(model)
        print(f"[DNN-{variant}] ep {epoch+1:02d}/{EPOCHS} n={len(ep_idx)} "
              f"train_acc={hist['accuracy'][-1]:.4f} loss={hist['loss'][-1]:.4f} | "
              f"val_acc={v_acc:.4f} val_f1={v_f1:.4f} val_loss={v_loss:.4f} | lr={lr:.2e}")

        improved = (v_f1 > best_f1 + 1e-4) or (abs(v_f1 - best_f1) <= 1e-4 and v_loss < best_loss - 1e-4)
        if improved:
            best_f1, best_loss, best_w, wait = max(v_f1, best_f1), v_loss, model.get_weights(), 0
        else:
            wait += 1
            if wait % LR_PATIENCE == 0:
                set_lr(model, max(lr * 0.3, 1e-7))
            if wait >= PATIENCE:
                print(f"[DNN-{variant}] early stopping (best val macro-F1 = {best_f1:.4f})")
                break
    model.set_weights(best_w)
    return model, hist, {"val_f1": float(best_f1), "val_loss": float(best_loss), "epochs": len(hist["loss"])}

def group_mean_accuracy(te, y_pred):
    df = pd.DataFrame({"g": groups[te], "ok": (y_pred == labels[te]).astype(float)})
    return df.groupby("g")["ok"].mean().mean()

def save_artifacts(tag, fold, y_true, y_pred, hist):
    rep = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)
    with open(f"{OUTPUT_DIR}/{tag}_fold{fold}_classification_report.txt", "w") as f:
        f.write(rep)
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    plt.figure(figsize=(8, 6), dpi=300)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {tag} (fold {fold})")
    plt.ylabel("True Label"); plt.xlabel("Predicted Label"); plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{tag}_fold{fold}_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()
    plt.figure(figsize=(18, 5), dpi=300)
    plt.subplot(1, 3, 1)
    plt.plot(hist["accuracy"], label="Train", lw=2); plt.plot(hist["val_accuracy"], label="Val", lw=2)
    plt.title(f"{tag} - Accuracy"); plt.xlabel("Epoch"); plt.legend()
    plt.subplot(1, 3, 2)
    plt.plot(hist["loss"], label="Train (weighted, smoothed)", lw=2); plt.plot(hist["val_loss"], label="Val", lw=2)
    plt.title(f"{tag} - Loss"); plt.xlabel("Epoch"); plt.legend()
    plt.subplot(1, 3, 3)
    plt.plot(hist["val_f1"], label="Val macro-F1", lw=2)
    plt.title(f"{tag} - Val macro-F1"); plt.xlabel("Epoch"); plt.legend()
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{tag}_fold{fold}_curves.png", dpi=300, bbox_inches="tight")
    plt.close()

def train_and_evaluate(variant, fold, tr, va, te):
    cfg = dict(variant=variant, DROPOUT=DROPOUT, LABEL_SMOOTH=LABEL_SMOOTH, SPG=SAMPLES_PER_GROUP, SIZE=SIZE)
    rhash = hashlib.md5(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:6]
    res_path = f"{OUTPUT_DIR}/DNN-{variant}_{rhash}_fold{fold}_result.json"
    pred_path = f"{OUTPUT_DIR}/DNN-{variant}_fold{fold}_test_pred.npy"
    if os.path.exists(res_path) and os.path.exists(pred_path):
        print(f"\n[DNN-{variant}] finished run found, loading it (delete {res_path} to retrain).")
        return json.load(open(res_path)), np.load(pred_path)
    print(f"\n{'='*60}\nFold {fold} - DNN [{variant}] @ {SIZE}px\n{'='*60}")
    t0 = time.time()
    model, hist, info = train_model(variant, tr, va)
    y_pred = model.predict(make_ds(te, SIZE), verbose=0).argmax(1)      # test touched here only
    y_true = labels[te]
    res = dict(model=f"DNN-{variant}", fold=fold, **info,
               test_acc=float(accuracy_score(y_true, y_pred)),
               test_macro_f1=float(f1_score(y_true, y_pred, average="macro")),
               test_group_acc=float(group_mean_accuracy(te, y_pred)),
               params=int(model.count_params()), minutes=(time.time() - t0) / 60)
    np.save(pred_path, y_pred)
    np.save(f"{OUTPUT_DIR}/DNN-{variant}_fold{fold}_test_idx.npy", te)
    with open(res_path, "w") as f:
        json.dump(res, f)
    save_artifacts(f"DNN-{variant}", fold, y_true, y_pred, hist)
    print(f">>> DNN-{variant}: VAL f1={info['val_f1']*100:.2f}% | TEST acc={res['test_acc']*100:.2f}%  "
          f"macro-F1={res['test_macro_f1']*100:.2f}%  group-acc={res['test_group_acc']*100:.2f}%  "
          f"({res['minutes']:.1f} min)")
    del model; tf.keras.backend.clear_session(); gc.collect()
    return res, y_pred

# ------------------------------------------------------------
# 7. RUN DNN
# ------------------------------------------------------------
dnn_rows = []
for fold in range(FOLDS_TO_RUN):
    tr, va, te = folds[fold]
    check_no_leak(tr, va, te)
    print(f"\n{'#'*60}\nFOLD {fold+1}/{FOLDS_TO_RUN}: train={len(tr)} val={len(va)} test={len(te)} (no group overlap)\n{'#'*60}")
    for variant in DNN_VARIANTS:
        res, _ = train_and_evaluate(variant, fold, tr, va, te)
        dnn_rows.append(res)

pd.DataFrame(dnn_rows).to_csv(f"{OUTPUT_DIR}/dnn_all_variants_results.csv", index=False)
print("\nDNN VARIANTS (both reported; the table below uses DNN_FINAL = '%s')" % DNN_FINAL)
print(pd.DataFrame(dnn_rows)[["model", "fold", "val_f1", "test_acc", "test_macro_f1",
                              "test_group_acc", "epochs", "params", "minutes"]].round(4).to_string(index=False))

# ------------------------------------------------------------
# 8. COMBINED SUMMARY: FF-CNN + ResNet50 + InceptionV3 + DNN
# ------------------------------------------------------------
def pretrained_recipe(name):
    p = f"{PRETRAINED_DIR}/all_variants_results.csv"
    if os.path.exists(p):
        df = pd.read_csv(p)
        r = df[(df["model"] == name) & (df["final"] == True)]
        if len(r): return r.iloc[0]["recipe"]
    return "strict"

def pred_path_for(model, fold):
    if model == "FF-CNN":
        return f"{FFCNN_DIR}/FF-CNN_fold{fold}_test_pred.npy", "as trained in Script 1"
    if model in ("ResNet50", "InceptionV3"):
        rec = pretrained_recipe(model)
        return f"{PRETRAINED_DIR}/{model}_{rec}_fold{fold}_test_pred.npy", f"{rec} recipe"
    return f"{OUTPUT_DIR}/DNN-{DNN_FINAL}_fold{fold}_test_pred.npy", f"{DNN_FINAL} variant"

def count_params(model):
    if model == "FF-CNN":
        inp = layers.Input(shape=(SIZE, SIZE, 3)); x = inp
        for f, k in [(32, 5), (64, 5), (128, 3), (256, 3), (512, 3)]:
            x = layers.Conv2D(f, k, padding="same")(x); x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x); x = layers.MaxPooling2D(2)(x)
        x = layers.GlobalAveragePooling2D()(x); x = layers.Dense(256, activation="relu")(x)
        return Model(inp, layers.Dense(NUM_CLASSES, activation="softmax")(x)).count_params()
    if model == "ResNet50":
        return ResNet50(weights=None, include_top=False).count_params() + 2048 * NUM_CLASSES + NUM_CLASSES
    if model == "InceptionV3":
        return InceptionV3(weights=None, include_top=False).count_params() + 2048 * NUM_CLASSES + NUM_CLASSES
    return int(next(r["params"] for r in dnn_rows if r["model"] == f"DNN-{DNN_FINAL}"))

def mcnemar_test(y_true, p1, p2, n1, n2):
    c1, c2 = (p1 == y_true), (p2 == y_true)
    b = int(np.sum(c1 & ~c2)); c = int(np.sum(~c1 & c2))
    if b + c == 0:
        return f"McNemar ({n1} vs {n2}): no discordant pairs"
    if b + c < 25:
        p, kind = binomtest(min(b, c), b + c, 0.5).pvalue, "exact"
    else:
        p, kind = 1 - chi2.cdf((abs(b - c) - 1) ** 2 / (b + c), 1), "chi2 (continuity-corrected)"
    return f"McNemar ({n1} vs {n2}): b={b}, c={c}, p={p:.4g} [{kind}]"

ALL_MODELS = ["FF-CNN", "ResNet50", "InceptionV3", "DNN"]
per_fold, notes, mcnemar_lines = [], {}, []

for fold in range(FOLDS_TO_RUN):
    te = folds[fold][2]
    y_true = labels[te]
    preds = {}
    for m in ALL_MODELS:
        p, note = pred_path_for(m, fold)
        if not os.path.exists(p):
            print(f"[summary] missing {p} - skipping {m} for fold {fold}")
            continue
        pr = np.load(p)
        if len(pr) != len(te):
            print(f"[summary] {m}: prediction length {len(pr)} != test size {len(te)}; skipping"); continue
        preds[m], notes[m] = pr, note
        per_fold.append(dict(model=m, fold=fold,
                             acc=accuracy_score(y_true, pr) * 100,
                             macro_f1=f1_score(y_true, pr, average="macro") * 100,
                             group_acc=group_mean_accuracy(te, pr) * 100))
    if "FF-CNN" in preds:
        for m in ALL_MODELS[1:]:
            if m in preds:
                mcnemar_lines.append(f"[fold {fold}] " + mcnemar_test(y_true, preds["FF-CNN"], preds[m], "FF-CNN", m))

pf = pd.DataFrame(per_fold)
pf.to_csv(f"{OUTPUT_DIR}/combined_per_fold.csv", index=False)

rows = []
for m in ALL_MODELS:
    d = pf[pf["model"] == m]
    if d.empty: continue
    sd = lambda s: s.std(ddof=1) if len(s) > 1 else 0.0
    rows.append({"Model": m, "Setting": notes.get(m, ""),
                 "Accuracy %": f"{d.acc.mean():.2f} ± {sd(d.acc):.2f}",
                 "Macro-F1 %": f"{d.macro_f1.mean():.2f} ± {sd(d.macro_f1):.2f}",
                 "Group-avg acc %": f"{d.group_acc.mean():.2f} ± {sd(d.group_acc):.2f}",
                 "Params (M)": round(count_params(m) / 1e6, 2),
                 "Folds": len(d)})
combined = pd.DataFrame(rows)
combined.to_csv(f"{OUTPUT_DIR}/combined_summary.csv", index=False)

print("\n" + "=" * 90 + "\nCOMBINED SUMMARY - held-out groups (all four models, identical test sets)\n" + "=" * 90)
print(combined.to_string(index=False))
print("\nMcNEMAR (FF-CNN vs others; p-values optimistic because images in a group are correlated)")
for line in mcnemar_lines: print(" ", line)

with open(f"{OUTPUT_DIR}/combined_summary.json", "w") as f:
    json.dump({"table": rows, "mcnemar": mcnemar_lines,
               "dnn_final_variant": DNN_FINAL, "n_images": int(N), "n_groups": N_GROUPS}, f, indent=2)

try:
    from google.colab import files
    shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
    print("\nDownloading results zip...")
    files.download(f"{OUTPUT_DIR}.zip")
except ImportError:
    print("Not in Colab - results are in", OUTPUT_DIR)

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision ON
Gathering image paths and labels...
58546 images, 5 classes: ['CCLD', 'CLM', 'HCL', 'PCLD', 'RCLD']
Loaded cached hashes from Paper_Results_v2/phash_cache.npy
58546 images -> 1110 groups (avg 52.74 images per original photo)

############################################################
FOLD 1/1: train=37203 val=8005 test=13338 (no group overlap)
############################################################

Fold 0 - DNN [stem] @ 224px
[DNN-stem] ep 01/40 n=11635 train_acc=0.6912 loss=1.0699 | val_acc=0.5973 val_f1=0.4449 val_loss=1.4844 | lr=1.00e-03
[DNN-stem] ep 02/40 n=11635 train_acc=0.7841 loss=0.8103 | val_acc=0.8657 val_f1=0.8308 val_loss=0.3621 | lr=1.00e-03
[DNN-stem] ep 03/40 n=11635 train_acc=0.8162 loss=0.7495 | val_acc=0.8720 val_f1=0.8133 val_loss=0.3599 | lr=1.00e-03
[DNN-stem] ep 04/40 n=11635 train_acc=0.8434 loss=0.6953 | val_acc=0.9088 val_f1=0.8721 val_loss=0.3127 | lr=1.00e-03

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>